In [1]:
from pathlib import Path
import kagglehub

# Download latest version
path = kagglehub.dataset_download("changheonkim/iam-trocr")
path = Path(path)/"IAM"
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'iam-trocr' dataset.
Path to dataset files: /kaggle/input/iam-trocr/IAM


In [2]:
import os

# Assuming 'path' variable holds the base directory from kagglehub.dataset_download
# If not, please replace 'path' with the correct directory string, e.g., '/content/IAM'
if 'path' in globals():
    print(f"Listing directories in: {path}")
    # Use a shell command to list only directories recursively, and sort them
    !ls {path/"image"}

Listing directories in: /kaggle/input/iam-trocr/IAM
c04-110-00.jpg	e06-070-02.jpg	 g07-000b-00.jpg  n02-157-05.jpg
c04-110-01.jpg	e06-070-03.jpg	 g07-000b-01.jpg  n02-157-06.jpg
c04-110-02.jpg	e06-070-04.jpg	 g07-000b-02.jpg  n02-157-07.jpg
c04-110-03.jpg	e06-070-05.jpg	 g07-000b-03.jpg  n02-157-08.jpg
c04-116-00.jpg	e06-070-06.jpg	 g07-000b-04.jpg  n03-038-00.jpg
c04-116-01.jpg	e06-070-07.jpg	 g07-000b-05.jpg  n03-038-01.jpg
c04-116-02.jpg	e06-070-08.jpg	 g07-000b-06.jpg  n03-038-02.jpg
c04-116-03.jpg	e06-070-09.jpg	 g07-000b-07.jpg  n03-038-03.jpg
c04-134-00.jpg	f04-032-00.jpg	 g07-000b-08.jpg  n03-038-04.jpg
c04-134-01.jpg	f04-032-01.jpg	 g07-000b-09.jpg  n03-038-05.jpg
c04-134-02.jpg	f04-032-02.jpg	 g07-079a-00.jpg  n03-038-06.jpg
c04-134-03.jpg	f04-032-03.jpg	 g07-079a-01.jpg  n03-064-00.jpg
c04-134-04.jpg	f04-032-04.jpg	 g07-079a-02.jpg  n03-064-01.jpg
c04-134-05.jpg	f04-032-05.jpg	 g07-079a-03.jpg  n03-064-02.jpg
c04-134-06.jpg	f04-032-06.jpg	 g07-079a-04.jpg  n03-064-03.jpg
c04

In [3]:
import glob

# Assuming 'path' is defined and points to the base directory of the dataset
# The images are located in the 'image' subdirectory relative to 'path'
image_directory = path / "image"

# Use glob to find all .jpg files in the image directory
image_paths = sorted(glob.glob(str(image_directory / "*.jpg")))

print(f"Found {len(image_paths)} images in the dataset.")
print("First 5 image paths:")
for i, img_path in enumerate(image_paths[:5]):
    print(f"  {i+1}: {img_path}")

Found 2915 images in the dataset.
First 5 image paths:
  1: /kaggle/input/iam-trocr/IAM/image/c04-110-00.jpg
  2: /kaggle/input/iam-trocr/IAM/image/c04-110-01.jpg
  3: /kaggle/input/iam-trocr/IAM/image/c04-110-02.jpg
  4: /kaggle/input/iam-trocr/IAM/image/c04-110-03.jpg
  5: /kaggle/input/iam-trocr/IAM/image/c04-116-00.jpg


In [13]:
from pathlib import Path
import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms

class IAMImageDataset(Dataset):
    def __init__(self, image_paths, labels_dict, transform=None):
        self.image_paths = image_paths
        self.labels_dict = labels_dict
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        file_name = Path(img_path).stem
        text = self.labels_dict.get(file_name, "")

        return image, text


# Transformações
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 🔹 Se você ainda não criou os labels:
labels_dict = {}  # substitua depois pelos textos reais

# Criar dataset
iam_dataset = IAMImageDataset(
    image_paths=image_paths,
    labels_dict=labels_dict,
    transform=transform
)

print(f"Number of samples in the dataset: {len(iam_dataset)}")

# Testar um exemplo
image, text = iam_dataset[0]
print(f"Shape da imagem: {image.shape}")
print(f"Texto associado: '{text}'")

Number of samples in the dataset: 2915
Shape da imagem: torch.Size([3, 224, 224])
Texto associado: ''


In [14]:
from transformers import DonutProcessor, VisionEncoderDecoderModel
import torch

processor = DonutProcessor.from_pretrained("naver-clova-ix/donut-base")
model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie decoder.model.decoder.embed_tokens.weight to decoder.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


VisionEncoderDecoderModel(
  (encoder): DonutSwinModel(
    (embeddings): DonutSwinEmbeddings(
      (patch_embeddings): DonutSwinPatchEmbeddings(
        (projection): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
      )
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): DonutSwinEncoder(
      (layers): ModuleList(
        (0): DonutSwinStage(
          (blocks): ModuleList(
            (0): DonutSwinLayer(
              (layernorm_before): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
              (attention): DonutSwinAttention(
                (self): DonutSwinSelfAttention(
                  (query): Linear(in_features=128, out_features=128, bias=True)
                  (key): Linear(in_features=128, out_features=128, bias=True)
                  (value): Linear(in_features=128, out_features=128, bias=True)
                  (dropout): Dropout(p=0.0, inplace=False)
                )

In [35]:
def predict(image):
    pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)

    # prompt inicial obrigatório no Donut
    task_prompt = "<s>"

    decoder_input_ids = processor.tokenizer(
        task_prompt,
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids.to(device)

    with torch.no_grad():
        outputs = model.generate(
            pixel_values,
            decoder_input_ids=decoder_input_ids,
            max_length=128,
            early_stopping=True
        )

    prediction = processor.batch_decode(outputs, skip_special_tokens=True)[0]
    return prediction

In [18]:
gt_file = path / "gt_test.txt"

with open(gt_file, "r", encoding="utf-8") as f:
    for i in range(5):
        print(f.readline())

c04-110-00.jpg	Become a success with a disc and hey presto ! You're a star ... . Rolly sings with

c04-110-01.jpg	assuredness " Bella Bella Marie " ( Parlophone ) , a lively song that changes tempo mid-way .

c04-110-02.jpg	I don't think he will storm the charts with this one , but it's a good start .

c04-110-03.jpg	CHRIS CHARLES , 39 , who lives in Stockton-on-Tees , is an accountant .

c04-116-00.jpg	He is also a director of a couple of garages . And he finds time as well to be a lyric



In [36]:
gt_file = path / "gt_test.txt"

labels_dict = {}

with open(gt_file, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")  # ← usar TAB

        if len(parts) == 2:
            image_name, text = parts

            image_id = Path(image_name).stem  # remove .jpg
            labels_dict[image_id] = text

print("Total labels:", len(labels_dict))
print("Primeiras 5 chaves:", list(labels_dict.keys())[:5])

Total labels: 2915
Primeiras 5 chaves: ['c04-110-00', 'c04-110-01', 'c04-110-02', 'c04-110-03', 'c04-116-00']


In [37]:
iam_dataset = IAMImageDataset(
    image_paths=image_paths,
    labels_dict=labels_dict,
    transform=None  # importante se estiver usando processor
)

In [39]:
image, text = iam_dataset[0]
print("GT:", text)
print("Pred:", predict(image))

GT: Become a success with a disc and hey presto ! You're a star ... . Rolly sings with


The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Pred: 


In [22]:
!pip install jiwer

In [23]:
from jiwer import wer, cer

In [33]:
def evaluate(dataset, num_samples=100):
    total_wer = 0
    total_cer = 0

    for i in range(num_samples):
        image, ground_truth = dataset[i]

        prediction = predict(image)

        total_wer += wer(ground_truth, prediction)
        total_cer += cer(ground_truth, prediction)

        if i < 5:
            print("\n--- Exemplo ---")
            print("GT :", ground_truth)
            print("Pred:", prediction)

    print("\n====================")
    print("Average WER:", total_wer / num_samples)
    print("Average CER:", total_cer / num_samples)

In [34]:
evaluate(iam_dataset, num_samples=50)


--- Exemplo ---
GT : Become a success with a disc and hey presto ! You're a star ... . Rolly sings with
Pred: 

--- Exemplo ---
GT : assuredness " Bella Bella Marie " ( Parlophone ) , a lively song that changes tempo mid-way .
Pred: 

--- Exemplo ---
GT : I don't think he will storm the charts with this one , but it's a good start .
Pred: 

--- Exemplo ---
GT : CHRIS CHARLES , 39 , who lives in Stockton-on-Tees , is an accountant .
Pred: 

--- Exemplo ---
GT : He is also a director of a couple of garages . And he finds time as well to be a lyric
Pred: 


KeyboardInterrupt: 